# Kiểm chứng V-JEPA 2 ViT-L trên EK100: 32.7 mean-class R@5 (action)

**Cài đặt Kaggle:** Accelerator = **GPU T4 x2**, Internet = **On**, thêm dataset EK100 qua *Add Input*.

Quy trình: (1) cài đặt, (2) tải checkpoint của Meta, (3) kiểm tra dataset, (4) benchmark tốc độ,
(5) chạy eval trên 2 GPU (chạy tiếp được nếu bị ngắt), (6) tính metric.
Log/logits lưu ở `/kaggle/working/logits`. Khi phiên hết giờ: *Save Version*, rồi thêm output đó làm input
cho phiên sau và copy lại vào `/kaggle/working/logits`; script sẽ bỏ qua các clip đã xong.

In [ ]:
BRANCH = "claude/vjepa2-verify-improve-dsfgtg"
VJEPA2_COMMIT = "204698b45b3712590f06245fbfba32d3be539812"  # version the code was checked against
!git clone -q https://github.com/1Tyson/Testing-And-Research-VJEPA.git -b $BRANCH /kaggle/working/repo
!git clone -q https://github.com/facebookresearch/vjepa2.git /kaggle/working/vjepa2 && cd /kaggle/working/vjepa2 && git checkout -q $VJEPA2_COMMIT
!pip install -q decord webdataset timm einops
import os
os.environ["VJEPA2_ROOT"] = "/kaggle/working/vjepa2"
R = "/kaggle/working/repo"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Checkpoints (~1.3GB encoder+predictor, probe) -> /tmp so they do not count toward the 20GB output limit
!mkdir -p /tmp/ckpt
!wget -q -nc -P /tmp/ckpt https://dl.fbaipublicfiles.com/vjepa2/vitl.pt
!wget -q -nc -P /tmp/ckpt https://dl.fbaipublicfiles.com/vjepa2/evals/ek100-vitl-256.pt
!ls -lh /tmp/ckpt

In [ ]:
# Official annotations (the full train csv is required: probe output classes are derived from it)
!git clone -q --depth 1 https://github.com/epic-kitchens/epic-kitchens-100-annotations.git /kaggle/working/ek100-ann
!ls /kaggle/input; find /kaggle/input -maxdepth 3 | head -30

In [ ]:
# >>> Set this to the folder that contains the EK100 videos (searched recursively for *.MP4)
VIDEO_ROOT = "/kaggle/input/<dataset-slug>"
!python $R/scripts/prepare_ek100.py --video_root $VIDEO_ROOT \
    --train_csv /kaggle/working/ek100-ann/EPIC_100_train.csv \
    --val_csv /kaggle/working/ek100-ann/EPIC_100_validation.csv \
    --out_dir /kaggle/working/work --probe_videos
# Check the "warnings" list above: missing videos or non-50/60 fps videos change the result.

## Benchmark tốc độ (20 clip trên 1 GPU)
Dùng số clip/giây để ước lượng thời gian chạy toàn bộ tập val (~9.6k clip). Nếu clip/s thấp mà GPU-util thấp
(xem `nvidia-smi`), thì bottleneck là decode video trên CPU.

In [ ]:
!python $R/scripts/eval_ek100.py --work_dir /kaggle/working/work --out_dir /tmp/bench \
    --vitl_ckpt /tmp/ckpt/vitl.pt --probe_ckpt /tmp/ckpt/ek100-vitl-256.pt \
    --device cuda:0 --dtype fp16 --max_clips 20 --save_every 10 2>&1 | grep -vE "DEBUG|^ |Warning" | tail -8

## Eval đầy đủ trên 2×T4
Mỗi GPU xử lý một nửa số video. Có thể chạy lại cell này bất cứ lúc nào, script sẽ tiếp tục từ chỗ đã dừng.
Thêm `--save_feats_pool 4` nếu muốn lưu feature (khoảng 5GB cho tập val) để dùng cho giai đoạn nghiên cứu.

In [ ]:
import subprocess
procs = []
for gpu in (0, 1):
    cmd = (f"python {R}/scripts/eval_ek100.py --work_dir /kaggle/working/work --out_dir /kaggle/working/logits "
           f"--vitl_ckpt /tmp/ckpt/vitl.pt --probe_ckpt /tmp/ckpt/ek100-vitl-256.pt "
           f"--device cuda:{gpu} --shard_id {gpu} --num_shards 2 --dtype fp16 --num_workers 2")
    procs.append(subprocess.Popen(cmd + f" > /kaggle/working/eval_gpu{gpu}.log 2>&1", shell=True))

In [ ]:
# Theo dõi tiến độ (chạy lại cell này)
!tail -n 2 /kaggle/working/eval_gpu0.log /kaggle/working/eval_gpu1.log
print([p.poll() for p in procs])  # [0, 0] = xong

In [ ]:
!python $R/scripts/compute_metrics.py --work_dir /kaggle/working/work \
    --logits_dir /kaggle/working/logits/official --out /kaggle/working/verify_ek100_vitl.json

### Đọc kết quả
- **official**: đếm clip giống như lần chạy inference 64 GPU của Meta (có clip được đếm 2 lần, có clip bị bỏ qua).
  Con số này nên khớp 32.7.
- **clean**: mỗi clip đếm đúng 1 lần. Đây là con số "đúng" nên báo cáo trong bài.
- Nếu lệch > 0.5 điểm: xem `dataset_report.json` (thiếu video? fps khác?), số clip `failed` trong log, và `fp32_retries`.